# Multi-period RAROC — fixture math walk-through

*Task 1.1 — Phase 1 Q1 of the Credenda plan (`PLAN.md` §V).*

This notebook documents how the three reference fixtures shipped in
`tests/fixtures/` were computed. The fixtures are the **golden** the
period engine (built in Task 1.2) must reproduce within the tolerances
in `docs/engine/multiperiod-spec.md` §10:

- per-period RAROC: 0.5 bp absolute
- NPV total cost: 0.1% relative
- per-period FPE: 0.5% relative
- effective spread: 0.5 bp absolute
- single-period parity (1-period schedule = today's calculator): 1e-12

The math is computed **independently** of `raroc_engine/calculator.py`
— a separate Python implementation in
`tests/fixtures/build_fixtures.py`. This is what makes the fixtures
*reference* fixtures rather than self-tests.

## 1. Per-period formulas

For each period $i$ with `commitment` $C_i$, `avg_drawn` $D_i$,
`remaining_maturity_years` $M_i$, year-fraction $\Delta t_i$ (1.0 for
every Q1 fixture period), bank static parameters
$\{s, f_{\text{commit}}, \text{PD}, \text{GRR}, \text{cost\_ratio},
\tau, r_f\}$:

$$
\begin{aligned}
\text{revenue}_i      &= s \cdot D_i \cdot \Delta t_i + f_{\text{commit}} \cdot (C_i - D_i) \cdot \Delta t_i + \text{period\_fees}_i\\
\text{cost}_i         &= \text{revenue}_i \cdot \text{cost\_ratio}\\
\text{exposure}_i     &= 0.25 D_i + 0.75 C_i \qquad \text{(confirmed MLT CCF)}\\
\text{LGD}            &= \max(1 - \text{GRR}, \text{LGD}_\text{floor})\\
R                     &= 0.12 \cdot \frac{1 + e^{-50 \text{PD}} - 2 e^{-50}}{1 - e^{-50}}\\
b                     &= (0.11852 - 0.05478 \cdot \ln \text{PD})^2\\
z                     &= \sqrt{\tfrac{1}{1-R}} \cdot \Phi^{-1}(\text{PD}) + \sqrt{\tfrac{R}{1-R}} \cdot \Phi^{-1}(0.999)\\
K_\text{IRB}         &= \text{LGD} \cdot (\Phi(z) - \text{PD}) \cdot \frac{1 + (M_i - 2.5) b}{1 - 1.5 b}\\
K_\text{floor}       &= \text{output\_floor\_pct} \cdot \text{SA\_RW(PD)} / 12.5\\
K_i                  &= \max(K_\text{IRB}, K_\text{floor})\\
\text{FPE}_i         &= \text{exposure}_i \cdot K_i\\
\text{EL}_i          &= \text{exposure}_i \cdot \text{PD} \cdot (1 - \text{GRR}) \cdot \Delta t_i\\
\text{RAROC}_i       &= (1 - \tau) \left( \frac{\text{revenue}_i - \text{cost}_i - \text{EL}_i}{\text{FPE}_i} + r_f \right)
\end{aligned}
$$

Discount factor under discrete-annual Act/365F (D-0003 §3):

$$
\text{DF}_i = (1 + r_\text{disc})^{-t_{\text{end},i}}
$$

Aggregates:

$$
\begin{aligned}
\text{NPV borrower cost}    &= \sum_i \text{revenue}_i \cdot \text{DF}_i\\
\text{NPV drawn balance}    &= \sum_i D_i \cdot \Delta t_i \cdot \text{DF}_i\\
\text{effective spread}      &= \frac{\text{NPV borrower cost}}{\text{NPV drawn balance}}\\
\overline{\text{RAROC}}      &= \frac{\sum_i \text{RAROC}_i \cdot \text{FPE}_i \cdot \Delta t_i}{\sum_i \text{FPE}_i \cdot \Delta t_i}
\end{aligned}
$$

The SA risk-weight is the Basel III SA step function for corporates
(BIS d424 Table 5) implemented in
`raroc_engine.calculator.RAROCCalculator._standardised_risk_weight`.

## 2. Setup

In [ ]:
import os, sys
# Resolve the fixtures dir regardless of which cwd the notebook is started from.
for candidate in ('../../tests/fixtures', 'tests/fixtures', '../tests/fixtures'):
    if os.path.exists(os.path.join(candidate, 'build_fixtures.py')):
        sys.path.insert(0, os.path.abspath(candidate))
        break
from build_fixtures import (
    EngineConfigSnapshot, DiscountSpec, PD_BY_RATING,
    fixture_rcf_5y, fixture_termloan_7y_amortising, fixture_projfin_10y_grace,
    compute_period, compute_aggregates, build_one,
)
import math
from scipy.stats import norm

cfg = EngineConfigSnapshot()
disc = DiscountSpec(kind='scalar', rate=cfg.risk_free_rate, day_count='Act/365F')
print('Risk-free rate :', cfg.risk_free_rate)
print('Tax rate       :', cfg.bank_tax_rate)
print('Cost ratio     :', cfg.default_cost_income_ratio)
print('Output floor   :', cfg.output_floor_pct)
print('PD floor       :', cfg.pd_floor)

## 3. Fixture 1 — Confirmed RCF, 5y, cleandown

Setup:
- 50M EUR commitment, 5 years
- Drawn: 35M/yr for years 1–3, 20M/yr for years 4–5 ("cleandown")
- Investment-grade mid-cap (rating **Baa2**, PD = 16 bp)
- Unsecured (GRR = 0); LGD = 1.0 (unsecured floor 25% doesn't bind)
- Spread = 150 bp, commitment fee = 25 bp on undrawn
- Upfront fee = 200,000 EUR (allocated to year 1)

Why this fixture matters: it covers the **revolver/cleandown** drawdown
shape — the most common pattern for RCFs in the Sabine persona's
portfolio (`PLAN.md` §II). The cleandown drops EAD in years 4–5, which
drops FPE and EL, but the maturity adjustment $b$ in $K$ falls faster
than the EAD, so RAROC actually rises across the cleandown — a
counterintuitive result the engine must show correctly.

In [ ]:
deal, periods = fixture_rcf_5y()
print(deal.description)
print()

rows = []
cum_t = 0.0
for p in periods:
    r = compute_period(p, deal, cfg, cum_t, disc.rate)
    rows.append(r)
    cum_t = r.t_end_years

agg = compute_aggregates(rows)

header = ['Per', 'Commit', 'Drawn', 'M', 'Rev', 'Cost', 'EAD', 'K', 'FPE', 'EL', 'RAROC%', 'DF']
print(' '.join(h.rjust(10) for h in header))
for r in rows:
    print(' '.join([
        f'{r.index:>10}',
        f'{r.commitment/1e6:>10,.0f}M',
        f'{r.avg_drawn/1e6:>10,.0f}M',
        f'{r.remaining_maturity_years:>10.0f}',
        f'{r.revenue/1000:>9,.1f}k',
        f'{r.cost/1000:>9,.1f}k',
        f'{r.exposure/1e6:>9,.2f}M',
        f'{r.K:>10.5f}',
        f'{r.fpe/1e6:>9,.3f}M',
        f'{r.el/1000:>9,.1f}k',
        f'{r.raroc*100:>10.3f}',
        f'{r.df:>10.5f}',
    ]))

### 3.1 Year-1 deep dive

Walk through every cell of the period-1 row by hand.

In [ ]:
p1 = periods[0]
r1 = rows[0]

# Revenue
spread_revenue = deal.spread * p1.avg_drawn * p1.dt_years
commit_revenue = deal.commitment_fee * (p1.commitment - p1.avg_drawn) * p1.dt_years
fees = p1.upfront_fee
print(f'spread × drawn × dt   = {deal.spread} × {p1.avg_drawn:,.0f} × {p1.dt_years} = {spread_revenue:,.2f}')
print(f'commit × undrawn × dt = {deal.commitment_fee} × {(p1.commitment - p1.avg_drawn):,.0f} × {p1.dt_years} = {commit_revenue:,.2f}')
print(f'upfront fee             = {fees:,.2f}')
print(f'revenue                 = {spread_revenue + commit_revenue + fees:,.2f}')
print(f'engine value            = {r1.revenue:,.2f}')
print()

# Exposure (CCF)
ead = 0.25 * p1.avg_drawn + 0.75 * p1.commitment
print(f'EAD (confirmed) = 0.25 × {p1.avg_drawn:,.0f} + 0.75 × {p1.commitment:,.0f} = {ead:,.2f}')
print(f'engine value    = {r1.exposure:,.2f}')
print()

# PD / R / b
pd_raw = PD_BY_RATING[deal.rating]
pd_v = max(pd_raw, cfg.pd_floor)
R = 0.12 * (1 + math.exp(-50*pd_v) - 2*math.exp(-50)) / (1 - math.exp(-50))
b = (0.11852 - 0.05478 * math.log(pd_v))**2
print(f'PD ({deal.rating}) raw     = {pd_raw}')
print(f'PD after Basel III floor = {pd_v}')
print(f'R                        = {R:.10f}')
print(f'b                        = {b:.10f}')
print()

# K
M = p1.remaining_maturity_years
lgd = max(1 - deal.global_grr, cfg.lgd_floor_unsecured)
z = math.sqrt(1/(1-R)) * norm.ppf(pd_v) + math.sqrt(R/(1-R)) * norm.ppf(0.999)
K_irb = lgd * (norm.cdf(z) - pd_v) * (1 + (M - 2.5) * b) / (1 - 1.5 * b)
K_floor = cfg.output_floor_pct * 0.75 / 12.5   # SA_RW=0.75 for PD in (0.0015, 0.0075]
K = max(K_irb, K_floor)
print(f'LGD = max(1 - {deal.global_grr}, {cfg.lgd_floor_unsecured}) = {lgd}')
print(f'z = sqrt(1/(1-R)) × Φ⁻¹(PD) + sqrt(R/(1-R)) × Φ⁻¹(0.999) = {z:.10f}')
print(f'K_IRB = LGD × (Φ(z) - PD) × (1+(M-2.5)b)/(1-1.5b)        = {K_irb:.10f}')
print(f'SA_RW for PD={pd_v} = 0.75 (PD ∈ (0.0015, 0.0075])')
print(f'K_floor = 0.55 × 0.75 / 12.5                              = {K_floor:.10f}')
print(f'K = max(K_IRB, K_floor)                                   = {K:.10f}')
print(f'engine K                                                  = {r1.K:.10f}')
print()

# FPE, EL
fpe = ead * K
el = ead * pd_v * (1 - deal.global_grr) * p1.dt_years
print(f'FPE = EAD × K = {fpe:,.2f}  (engine: {r1.fpe:,.2f})')
print(f'EL  = EAD × PD × (1-GRR) × dt = {el:,.2f}  (engine: {r1.el:,.2f})')
print()

# RAROC
raroc = (1 - cfg.bank_tax_rate) * ((r1.revenue - r1.cost - r1.funding_cost - el) / fpe + cfg.risk_free_rate)
print(f'RAROC = (1-τ) × ((rev - cost - fund - EL)/FPE + rfr)')
print(f'      = 0.75 × (({r1.revenue:,.0f} - {r1.cost:,.0f} - 0 - {el:,.0f}) / {fpe:,.0f} + 0.0325)')
print(f'      = {raroc:.10f}')
print(f'engine RAROC = {r1.raroc:.10f}')

### 3.2 Why RAROC rises after the cleandown

Look at period 5 vs period 4 — same drawn (20M), same commitment
(50M), so same revenue. But $M_4 = 2$ and $M_5 = 1$. The Basel
maturity adjustment $(1 + (M-2.5)b)/(1-1.5b)$ falls from
$1.333/0.667 \approx 2.00$ at $M=2$ to $1.000/0.667 \approx 1.50$
at $M=1$ — i.e. $K$ drops 25% while EL is unchanged. RAROC,
which is (profit − EL) / FPE + rfr, mechanically rises.

The Term-Sheet Doctor UI (Module A) will need to call this out:
a borrower offering a low spread on the **final year** of a long
facility is more expensive to the bank in *rate-of-return* terms
than the headline RAROC suggests — because the bank is releasing
capital fastest in the final year. Quoting period-1 RAROC alone
is misleading; the FPE-weighted aggregate is the right headline.

In [ ]:
for r in rows:
    mat_adj = (1 + (r.remaining_maturity_years - 2.5) * r.maturity_adj_b) / (1 - 1.5 * r.maturity_adj_b)
    print(f'Period {r.index}: M={r.remaining_maturity_years:.0f}, maturity_adj={mat_adj:.4f}, '
          f'K={r.K:.5f}, FPE={r.fpe/1e6:.3f}M, RAROC={r.raroc*100:.2f}%')
print()
print(f'FPE-weighted RAROC (headline) = {agg["fpe_weighted_raroc"]*100:.2f}%')
print(f'NPV borrower cost              = {agg["npv_borrower_cost"]:,.0f}')
print(f'Effective spread                = {agg["effective_spread_bp"]:.2f} bp')

## 4. Fixture 2 — Term loan, 7y, linear amortising

Setup:
- 70M EUR day-1 drawn (commitment = drawn each year)
- Linear amortisation: 10M/yr paydown to 0
- $\text{avg\_drawn}_i = (\text{start}_i + \text{end}_i)/2$:
  65M, 55M, 45M, 35M, 25M, 15M, 5M
- Crossover-credit borrower (rating **Baa3**, PD = 24 bp)
- Partially secured (GRR = 0.20); LGD = 0.80 (above the 10% secured floor)
- Spread = 175 bp, no commitment fee (fully drawn each year)
- Upfront fee = 350,000 EUR (year 1)

Why this fixture matters: it exercises **scheduled amortisation**
and **non-zero GRR**. The interesting numerical features are
(a) commitment and drawn move in lockstep, so there is never
any commit-fee revenue; (b) GRR=0.2 means LGD = 0.80, the secured
LGD floor of 0.10 doesn't bind; (c) the maturity adjustment shows
monotonic decline across all 7 periods.

In [ ]:
deal_tl, periods_tl = fixture_termloan_7y_amortising()
rows_tl, agg_tl, _, _ = build_one(deal_tl, periods_tl, cfg, disc, '/tmp', '/tmp')
print(deal_tl.description)
print()
header = ['Per', 'Drawn', 'M', 'Rev', 'EAD', 'K', 'FPE', 'EL', 'RAROC%', 'DF']
print(' '.join(h.rjust(10) for h in header))
for r in rows_tl:
    print(' '.join([
        f'{r.index:>10}',
        f'{r.avg_drawn/1e6:>9,.1f}M',
        f'{r.remaining_maturity_years:>10.0f}',
        f'{r.revenue/1000:>9,.1f}k',
        f'{r.exposure/1e6:>9,.2f}M',
        f'{r.K:>10.5f}',
        f'{r.fpe/1e6:>9,.3f}M',
        f'{r.el/1000:>9,.2f}k',
        f'{r.raroc*100:>10.3f}',
        f'{r.df:>10.5f}',
    ]))

print()
print(f'FPE-weighted RAROC = {agg_tl["fpe_weighted_raroc"]*100:.2f}%')
print(f'NPV borrower cost  = {agg_tl["npv_borrower_cost"]:,.0f}')
print(f'Effective spread    = {agg_tl["effective_spread_bp"]:.2f} bp')

## 5. Fixture 3 — Project finance, 10y, grace + amortise + bullet

Setup:
- 100M EUR commitment, 10 years
- Years 1–3 drawdown ramp (avg drawn 30M → 70M → 100M)
- Years 4–5 grace period (100M drawn, no principal repayment)
- Years 6–9 linear amortisation (drawn drops 100M → 20M)
- Year 10 bullet (20M, repaid at end-y10; avg drawn 20M)
- Project sponsor with crossover credit (rating **Ba1**, PD = 38 bp)
- Project-asset security (GRR = 0.30); LGD = 0.70
- Spread = 225 bp, commitment fee = 35 bp on undrawn
- Upfront fee = 1,000,000 EUR (year 1)

Why this fixture matters: it covers the **drawdown ramp + grace +
amortise + bullet** profile typical of infrastructure / real-estate
project finance. Years 1–2 have meaningful commit-fee revenue
(because the facility is undrawn during construction); years 3–5
have no commit revenue (fully drawn during grace); years 6–10
blend amortisation and a small bullet at maturity. RAROC moves
non-monotonically as commit-fee revenue interacts with declining
$M$.

In [ ]:
deal_pf, periods_pf = fixture_projfin_10y_grace()
rows_pf, agg_pf, _, _ = build_one(deal_pf, periods_pf, cfg, disc, '/tmp', '/tmp')
print(deal_pf.description)
print()
header = ['Per', 'Commit', 'Drawn', 'M', 'Rev', 'EAD', 'K', 'FPE', 'EL', 'RAROC%']
print(' '.join(h.rjust(10) for h in header))
for r in rows_pf:
    print(' '.join([
        f'{r.index:>10}',
        f'{r.commitment/1e6:>9,.0f}M',
        f'{r.avg_drawn/1e6:>9,.0f}M',
        f'{r.remaining_maturity_years:>10.0f}',
        f'{r.revenue/1000:>9,.1f}k',
        f'{r.exposure/1e6:>9,.2f}M',
        f'{r.K:>10.5f}',
        f'{r.fpe/1e6:>9,.3f}M',
        f'{r.el/1000:>9,.1f}k',
        f'{r.raroc*100:>10.3f}',
    ]))

print()
print(f'FPE-weighted RAROC = {agg_pf["fpe_weighted_raroc"]*100:.2f}%')
print(f'NPV borrower cost  = {agg_pf["npv_borrower_cost"]:,.0f}')
print(f'Effective spread    = {agg_pf["effective_spread_bp"]:.2f} bp')

## 6. Summary table — what the engine must reproduce

| Fixture | Periods | Total revenue (undisc) | NPV borrower cost | Effective spread (bp) | FPE-weighted RAROC |
|---|---|---|---|---|---|
| RCF 5y cleandown | 5 | 2,637,500 | 2,426,730 | 182.23 | 7.74% |
| Term loan 7y amortising | 7 | 4,637,500 | 4,257,395 | 190.14 | 8.45% |
| Project finance 10y grace | 10 | 17,040,000 | 14,646,804 | 259.35 | 7.83% |

Conformance check (Task 1.2/1.4 — engine + harness):

1. Load each YAML.
2. Pass `engine_config`, `deal`, `schedule`, `discount` to
   `period_engine.run(...)`.
3. For every period, assert per-cell match against
   `expected.per_period[i]` within the relevant tolerance.
4. For aggregates, assert match within `tolerances.npv_rel`
   (relative for NPVs, absolute-bp for spreads, absolute-bp for
   RAROC).
5. Single-period parity test: build a 1-period schedule with
   `dt_years=1.0` and run through `RAROCCalculator.calculate(...)` —
   the period engine output must match to 1e-12 on every field.

## 7. Cross-check: spot-check a single-period equivalence

If we collapse the RCF year-1 inputs into a single-period
`RAROCInput` and run `RAROCCalculator.calculate(...)`, we should get
the same RAROC the period engine produces for period 1 of the RCF
fixture (to 1e-12).

In [ ]:
from raroc_engine.models import RAROCInput
from raroc_engine.calculator import RAROCCalculator
from raroc_engine.config import EngineConfig
from raroc_engine.repository import Repository

calc_inp = RAROCInput(
    product_type='mlt_credit',
    rating='Baa2',
    global_grr=0.0,
    confirmed=True,
    initial_volume=50_000_000,
    average_volume=50_000_000,
    initial_drawn=35_000_000,
    average_drawn=35_000_000,
    initial_maturity=60,           # 5 years in months
    residual_maturity=60,           # period 1 has M=5y
    spread=0.0150,
    commitment_fee=0.0025,
    upfront_fee=200_000,
)
calc_cfg = EngineConfig(
    regime='basel3',
    risk_free_rate=cfg.risk_free_rate,
    bank_tax_rate=cfg.bank_tax_rate,
    funding_cost_bp=cfg.funding_cost_bp,
    default_cost_income_ratio=cfg.default_cost_income_ratio,
    output_floor_pct=cfg.output_floor_pct,
    pd_floor=cfg.pd_floor,
)
calc = RAROCCalculator(repository=Repository(), config=calc_cfg)
out = calc.calculate(calc_inp)
print(f'Single-period (existing calculator) RAROC = {out.raroc:.12f}')
print(f'Period engine fixture period-1 RAROC      = {rows[0].raroc:.12f}')
print(f'Difference                                 = {abs(out.raroc - rows[0].raroc):.2e}')

---

**Future agent reading this notebook:** if you change any input
parameter in `build_fixtures.py`, regenerate the YAML and Excel by
running:

```
python tests/fixtures/build_fixtures.py
```

Then re-run this notebook to refresh the summary table. The YAML
is the source of truth for the test harness; the Excel is the
human-readable cross-check; this notebook is the methodology trail.